# Verify, walked through: where every number in the paper comes from

This notebook runs the paper's verification section by section and explains each
step: what the number is, where the paper uses it, how it is computed, and what
could have gone wrong in computing it.

**Nothing here is a second implementation.** Every table below is printed by
`verify.py` itself — each cell calls its `main()` restricted to one section with
`--only`. The notebook adds commentary between the calls, and in a few places a
short *look inside* cell that recomputes a single number in plain numpy so the
arithmetic is visible. Those cells are illustrations; the checks are the
authority.

**Reading a check.** Every line has the same five columns:

| column | meaning |
|---|---|
| `CLAIM` | what is being checked, in words close to the paper's |
| `PAPER` | the value the paper prints, from `expected.json` |
| `RECOMPUTED` | the value computed now, from the shipped data |
| `TOL` | the tolerance; its reason is recorded in `expected.json` |
| `STATUS` | `PASS` if the two agree within it |

**Runtime.** The first section that needs text embeddings encodes about 150,000
generations once — minutes on a GPU, 15–30 on a CPU. The cache in `cache/` makes
every later cell fast.

In [ ]:
import sys, os, io, contextlib
import verify

CACHE = "cache"   # embeddings are computed once and reused by every later cell

def run(*sections):
    # Run verify.py's own main() on the given sections only, exactly as
    #   python verify.py --cache cache --only <sections>
    # would. main() ends with sys.exit(), which a notebook must catch.
    known = set(verify.SECTIONS) | set(getattr(verify, "FULL_ONLY", []))
    missing = [s for s in sections if s not in known]
    if missing:
        print(f"  this verify.py has no section {missing}; skipped")
    sections = [s for s in sections if s in known]
    if not sections:
        return
    sys.argv = ["verify.py", "--cache", CACHE, "--only", *sections]
    try:
        verify.main()
    except SystemExit as e:
        code = e.code if isinstance(e.code, int) else 1
        print("\n  -> all checks passed" if code == 0 else f"\n  -> {code} check group(s) FAILED — read the rows above")

print("verify.py loaded from", os.path.abspath(verify.__file__))
print("data folder:", os.path.abspath(verify.DATA))

## 1. The basis and its geometry (paper §3.1, §5, Table 5)

Each axis is the normalised difference between the mean representations of two
word lists (equation 1):

$$\mathbf d_c = \frac{\boldsymbol\mu_P-\boldsymbol\mu_N}{\lVert\boldsymbol\mu_P-\boldsymbol\mu_N\rVert},\qquad \boldsymbol\mu_P = \frac1{|P|}\sum_{w\in P}\mathbf x_w .$$

A word's representation $\mathbf x_w$ is the mean of its hidden states over the
emotion-bearing senses of that word, each sense itself an average over eight
natural sentences. Only senses labelled `EMO` in `labels.csv` enter.

Stacking the seven axes gives $\mathbf P$, and $\mathbf G=\mathbf P\mathbf P^\top$
is their Gram matrix: ones on the diagonal, cosines off it. From $\mathbf G$ the
section derives

* **mean and maximum |cos|** — 0.187 and 0.467 (intensity–antagonism);
* **condition number** — 5.49: well conditioned, far from the identity;
* **leak** of axis $i$, $\sqrt{\sum_{j\ne i}G_{ji}^2}$ — how much of a unit request
  lands on the other axes; up to four fifths of a unit for antagonism–peace;
* **iso-cost** $\sqrt{(\mathbf G^{-1})_{ii}}$ — the extra displacement needed to move
  one coordinate alone: 1.03 to 1.31;
* **request cost** $\sqrt{\mathbf l^\top\mathbf G^{-1}\mathbf l}/\lVert\mathbf l\rVert$ — from
  $1/\sqrt{\lambda_{\max}}$ to $1/\sqrt{\lambda_{\min}}$: 0.68 to 1.60;
* **Gram–Schmidt retention** (Table 6) — how much of each axis survives
  orthogonalisation in the order used;
* **the alternative seven** — the basis with drive, desire and security_alarm
  substituted: mean 0.293, condition 13.2.

**The interface** (§4, Table 4) is checked in the same run, from the shipped atlas
and with the page's own arithmetic: under raw steering a request $\mathbf l$
arrives as $\mathbf G\mathbf l$; under the dual basis as $\mathbf l$ itself. Tokens are
ranked by their atlas coordinates against $\mathbf G^{-1}\mathbf c$, and the Ekman
proximities are cosines under the metric $\mathbf G^{-1}$ — treating the oblique
basis as orthonormal here is precisely the error the paper measures.

*What went wrong once:* a superseded label file shifted every geometry number
(0.177 instead of 0.187) without raising an error. `verify.py` refuses it.

In [ ]:
run("geometry", "interface")

### Look inside: the Gram matrix from the raw vectors

The same construction in fifteen lines of numpy, for the primary model. It should
print 0.187, 0.467 and 5.49 — the first rows of the table above.

In [ ]:
import glob, csv, json, numpy as np
from collections import defaultdict

vp = sorted(glob.glob(os.path.join(verify.DATA, "vectors_layer", "*gemma-3-1b-it*.npz")))[0]
z = np.load(vp, allow_pickle=False)
labels = {r["key"]: (r.get("final", "").strip().upper() or r["heuristic"])
          for r in csv.DictReader(open(os.path.join(verify.DATA, "labels.csv"), encoding="utf-8"))}
probes = json.load(open(os.path.join(verify.DATA, "probes.json"), encoding="utf-8"))
AXES = ["valence", "heat", "arousal", "intensity", "antagonism_peace", "body_mind", "social_inner_outer"]

keys = [str(k) for k in z["keys"]]
emo = np.array([labels.get(k) == "EMO" for k in keys])             # emotion-bearing senses only
layers = [int(x) for x in np.asarray(z["layers"])]
X = z["vecs"][emo, layers.index(17), :].astype(np.float64)         # the paper layer
words = np.array([str(w).lower() for w in z["words"]])[emo]
acc = defaultdict(list)
for w, v in zip(words, X):
    acc[w].append(v)
wv = {w: np.mean(v, 0) for w, v in acc.items()}                    # one vector per word

def axis(k):
    mp = np.mean([wv[w.lower()] for w in probes[k]["pos"] if w.lower() in wv], 0)
    mn = np.mean([wv[w.lower()] for w in probes[k]["neg"] if w.lower() in wv], 0)
    d = mp - mn                                                      # equation 1
    return d / np.linalg.norm(d)

P = np.array([axis(k) for k in AXES])
G = P @ P.T
off = np.abs(G[~np.eye(7, dtype=bool)])
print(f"mean |cos| {off.mean():.3f}   max {off.max():.3f}   condition {np.linalg.cond(G):.2f}")

## 2. Is each axis real? The nulls and the validation (§3.3)

An axis defined by stipulation is not thereby real. Three tests ask whether the
direction reflects the construct or merely the words chosen.

**The null is built from words.** Any contrast between two groups of words
carries frequency, length and topic structure that models encode, so it looks
meaningful against *random vectors* whatever it captures. The null that matters
draws $n^+$ and $n^-$ random corpus words — the pole sizes of the axis under
test — builds a contrast exactly as in equation 1, and repeats 200 times. Its
95th percentile is 0.851 against 0.246 for a random-vector null: four of the
seven axes clear the second but not the first.

**Held-out pole words (leave-one-word-out).** Drop one word from each pole,
rebuild the axis from the rest, and ask whether the held-out pair orders
correctly — over every possible pair. The null permutes the pole assignment among
the same words. Accuracies run from 0.828 (body–mind, against its null of 0.781)
to 1.000.

**Pre-registered unseen words.** About ten words per axis that appear nowhere in
the corpus, each assigned a pole *before* any extraction (`predictions.csv`),
extracted separately and projected with the constant-offset property of §3.1.
Separation reaches AUC 0.891 to 1.000, and every axis clears its null in all nine
models.

In [ ]:
run("nulls", "validation")

## 3. How hard each model can be steered (§3.5, Table 3)

Every steering result is taken at the model's own **coherence edge** $c^*$: the
largest magnitude, as a fraction of the residual norm, at which steered text
stays fluent. A generation passes if its moving-average type–token ratio is at
least 70% of unsteered text's and it has no repeated run longer than three
tokens and no repeated cycle longer than two; $c^*$ is the last level before the
pass rate falls below 70% of the unsteered rate.

The edges themselves needed a GPU to measure, so here each $c^*$ is re-derived
from its recorded ladder — the pass rate at each of fifteen magnitudes — rather
than regenerated.

**Why it matters:** run at one common magnitude instead ($c = 0.08$ for every
model), the simultaneous-control experiment of §7.2 reads 54% where matched
magnitudes give 86%. That run predates the edge protocol; its fitted transfer
matrices ship with this package, and the 54% is recomputed from them with the
same rule as the main result.

In [ ]:
run("edges", "commonmag")

## 4. Part of the overlap is the word lists themselves (§5.1)

Twelve words appear in more than one list — *rage* in heat, arousal and
intensity; *anguish* and *despair* at the negative pole of valence and the
positive pole of intensity. A shared word pulls two directions together or apart
whatever the model does.

The check rebuilds every axis without them: the mean off-diagonal falls from
0.187 to 0.140. Is that just thinner lists? The null removes the *same number of
randomly chosen unshared words* instead, 500 times: that leaves the mean at 0.209,
and none of the 500 draws falls as low as 0.140. Heat–intensity and
valence–intensity fall to near zero; intensity–antagonism keeps two thirds of its
size.

In [ ]:
run("sharedwords")

## 5. What a linear model would show, and what geometry predicts (§7.3)

Two questions the later results depend on, answered before looking at any
generated text.

**The linear baseline.** Suppose the model responded perfectly linearly. What
would our pipeline then report? Two properties of the experiment move that answer
away from the textbook one: every cell's displacement is rescaled to the same
length (§3.5), which makes the applied displacement non-linear in the dial
settings; and the readout projects text onto encoder directions that overlap
among themselves (mean |cos| 0.160). The check computes the transfer matrices a
linear model would produce through exactly this pipeline.

**The geometric prediction.** How much should the dual basis improve
separability one dial at a time, if the response were purely geometric — the Gram
of the steering directions and the overlap of the readout directions? ×2.03 on
average; for Gemma-3-1B-it ×1.95. The observed gain (§8 below) is ×1.30. The
difference is the part of the response that is not geometric.

In [ ]:
run("linear", "geompred")

## 6. Does each axis steer its own emotion? (§6, Table 7, Figure 3)

From here on the checks read generated text: 90,720 generations in three arms.

**The measure.** Generate text with an axis high and with it low, embed both sets
with a sentence encoder that is none of the models under test, and compare the
displacement with the direction the same encoder assigns to the construct
(equation 8):

$$\Delta_\text{text}=\overline{E(\text{high})}-\overline{E(\text{low})},\quad
\Delta_\text{probe}=\overline{E(P)}-\overline{E(N)},\quad
\operatorname{align}(c)=\cos(\Delta_\text{text},\Delta_\text{probe}).$$

**The control.** Steering along *any* direction changes text, so every trial is
paired with one steering a random direction of matched norm through the identical
pipeline. The reported gain is align minus its control: +0.405 across the nine
models, $t(8)=13.7$, all 63 model–axis cells positive.

**Three artefacts excluded** (`robustness`): the steered word merely repeated —
collapse repeated token cycles first (gains move by at most 0.002); the effect
living in degenerate text — drop every looping or collapsed generation (at most
0.034, every axis still positive).

**Construct-specific, not a general emotional shift** (`crossboot`): project each
axis's $\Delta_\text{text}$ onto every other axis's $\Delta_\text{probe}$. The diagonal
is the largest entry in all seven rows (mean +0.478 against +0.059 off it), and
stays so in all 1,200 bootstrap resamples, with a smallest margin of +0.158.

*What went wrong once:* an embedding cache keyed on the first three texts
collided across axes and reported that diagonal as five of seven. The bootstrap,
which had no cache, returned seven of seven — which is why the cache now hashes
every text.

In [ ]:
run("steering", "robustness", "crossboot")

## 7. Three dials at once (§7.1–7.4, Tables 10–14)

**The design.** Every trial steers three axes in a full factorial: all eight
combinations of high and low. Regressing each cell's measured position on the
requested levels,

$$\mathbf p=\mathbf M\mathbf l+\mathbf b,\qquad \mathbf M\in\mathbb R^{3\times3},$$

gives a transfer matrix: the intended effects on its diagonal, cross-talk off it.

**Row convention: rows = construct.** In these matrices "the diagonal is the row
maximum" means *each construct is moved most by its own dial*. (The
superposition matrices of §8 below use the other convention; each check's label
says which.)

**The results.** 85.7% of rows per repetition against a chance of one in three;
91.5% with the two repetitions of each triple pooled. The pooled figure comes from
the analysis that also uses raw text and unit-normalised positions; pooling alone,
in this package's conventions, gives 90.9%. The pair table of §7.4 is computed
here too, from deduplicated text.

**The dual basis, per trial.** The same trials rerun with $\mathbf D=\mathbf G^{-1}\mathbf P$.
Decomposed, the symmetric off-diagonal falls 23.6% and the diagonal 20.3%, while
the antisymmetric part does not move (+0.7%): $\mathbf G$ is symmetric, so
$\mathbf G^{-1}$ can act only on the symmetric part. Per-trial separability moves by
×0.92 — a figure the paper reports and then sets aside, because on matrices built
from six generations per cell it measures noise more than steering (§8).

In [ ]:
run("control")

## 8. How far the text moves (§6.3, Table 9, Figures 5 and 8)

Alignment says *which way* text moves; these say *how far*. The displacement
projected onto the construct is expressed three ways:

* **pole fraction** — as a share of the distance between the construct's own
  poles in the encoder: 25.0% for intensity, 8.3% for isolation–belonging;
* **range fraction** — against the range the construct spans across the triple's
  own generations, so it does not depend on the word lists; it agrees with the
  pole fraction on ordering (ρ = 0.96);
* **Cohen's d** — in units of the spread across individual generations: above 0.7
  for every axis.

**The spread** is the honest floor of the method: across 1,080 matched pairs per
axis the median runs from 7.5% to 21.9% of the pole distance, every axis has pairs
that move the wrong way, and about one pair in five moves less than 5%.

In [ ]:
run("magnitude", "spread")

## 9. Superposition, and what the dual basis really does (§7.3, Table 13, Figure 7)

The paper's central result, checked from two generation sets: the three-dial runs
and a **single-dial** run on the same trials, stems and magnitudes, each cell
turning one axis alone.

**The test.** Under a linear response every three-dial cell is predicted exactly
by the single-dial ones, rescaling included:

$$\mathbf y(\mathbf l)-\mathbf y_0=\sum_i l_i\,\frac{\lVert\mathbf d_i\rVert}{\lVert\sum_j l_j\mathbf d_j\rVert}\,\mathbf R_i ,$$

with $\mathbf R_i$ the single-dial response of axis $i$. The prediction agrees with
observation at r = 0.76, against 0.72 for one repetition of the experiment
predicting the other: as good as the noise allows. **The response is linear.**

**Row convention here: rows = dial.**

**The dual basis, scored properly.** Per-trial matrices are noise-dominated, so
each dial's effect on each construct is first averaged over every trial in which
the pair appears. On these pooled matrices the dual basis improves separability
×1.30 one dial at a time (8 of 9 models), but only ×1.05 with three at once (7 of 9,
not significant) — and the single-dial runs predict ×1.06 for that, r = 0.74 across
models. Within the linear model the whole loss comes from holding the total
strength of a request fixed.

**What no geometric correction reaches.** In the pooled single-dial matrices the
antisymmetric part — one axis moving another more than it is moved back — is 29%
of the cross-talk; the geometry alone produces 6%.

In [ ]:
run("superposition")

## 10. A second encoder (`--full`; §6.2, Table 8)

Everything in §6 depends on the sentence encoder. This section repeats the
steering measurement with bge-large-en-v1.5, from a different training family:
mean gain +0.507 ($t(8)=14.1$), all 63 cells positive and of the same sign as
under the primary encoder, the cross-alignment diagonal maximal in all seven
rows, and the ordering of the axes preserved (ρ = 0.79, exact p = 0.048). It is
the slowest section on a CPU.

In [ ]:
run("encoder2")

## What this does not verify

The extraction that produced the vectors and the generation that produced the
texts: both need model weights and a GPU, and belong to the reproduction
repository. This notebook verifies the **analysis**, not the experiment.

## How this package was hardened

Three analysis bugs once produced plausible numbers rather than errors (paper,
Appendix E): generation files whose arm suffix was not stripped, so signal and
control never paired; model names containing a dot, split by a path parser, so
four of nine models went silently unchecked; and the cache collision of section 6.
Each was caught by an independent recomputation, not by inspection — which is
the reason this package exists.

To run everything at once, outside the notebook:

```
python verify.py --full --cache cache
```